# 01 · Data Collection & Exploration
**CSC17104 – In-Vehicle Coupon Recommendation**  
**Nhóm:** Trần Nhật Dương · Nguyễn Mạnh Hà · Hoàng Công Lý

> Starter notebook: tự điền giải thích, ghi nhận số liệu & biểu đồ, không có kết quả phân tích sẵn.

## 1. Data Collection *(Owner: Trần Nhật Dương)*

### 1.1 Nguồn dữ liệu, tác giả & giấy phép
- **Tên bộ dữ liệu:** *In-Vehicle Coupon Recommendation* — UCI Machine Learning Repository, Dataset ID **603**.
- **Liên kết & DOI:** <https://archive.ics.uci.edu/dataset/603/in+vehicle+coupon+recommendation> · DOI: `10.24432/C5GS4P`.
- **Giấy phép (License):** **Creative Commons Attribution 4.0 International (CC BY 4.0)** — cho phép sử dụng, chia sẻ và phân tích học thuật với điều kiện ghi nhận nguồn tác giả.
- **Bài báo gốc:** Wang, T., Rudin, C., Doshi-Velez, F., Liu, Y., Klampfl, E., & MacNeille, P. (2017). *A Bayesian framework for learning rule sets for interpretable classification*. *Journal of Machine Learning Research (JMLR)*, 18(70), 1–37 (2357–2393).

### 1.2 Phương thức thu thập & ý nghĩa một dòng dữ liệu
- **Phương thức khảo sát:** Dữ liệu được thu thập thông qua khảo sát trực tuyến trên nền tảng **Amazon Mechanical Turk (MTurk)**.
- **Ý nghĩa mỗi quan sát (1 dòng):** Mỗi dòng (`12.684` dòng × `26` cột gồm `25` đặc trưng và `1` nhãn mục tiêu `Y`) mô tả **một kịch bản lái xe giả định** (điểm đến, thời tiết, nhiệt độ, thời điểm trong ngày, người đi cùng, loại coupon, thời hạn, khoảng cách/hướng đi ước lượng, kèm thông tin nhân khẩu học và thói quen tự khai báo) cùng câu trả lời nhị phân `Y` (`1` = đồng ý nhận coupon, `0` = từ chối).
- **Phân biệt ý định và hành vi thực tế:** `Y=1` phản ánh **ý định chấp nhận trong tình huống giả định**, không phải giao dịch đổi coupon thành công ngoài đời thực.

### 1.3 Lý do chọn dữ liệu & các giới hạn/bias cần lưu ý
- **Lý do lựa chọn:** Đủ quy mô (`12.684` dòng ≥ `3.000`, `25` features ≥ `10`), sạch cấu trúc, có giấy phép CC BY 4.0 rõ ràng, và giàu ngữ cảnh hành trình — phù hợp để khai thác sâu cả phân tích thống kê có kiểm soát nhiễu (Q1–Q5) lẫn bài toán phân loại nhị phân (Q6).
- **Giới hạn & nguy cơ bias:**
  1. **Thiếu `participant_id`:** Nhiều kịch bản có thể do cùng một người tham gia khảo sát trả lời lặp lại, nhưng file công bố không cung cấp mã định danh cá nhân (ảnh hưởng đến giả định độc lập khi tính CI và chiến lược chia train/test ở Q6).
  2. **Sai lệch chọn mẫu MTurk & tự khai báo:** Người trả lời trên MTurk không đại diện hoàn toàn cho mọi nhóm tài xế; tần suất ghé quán (`Bar`, `CoffeeHouse`, ...) là dữ liệu tự nhớ lại.
  3. **Ràng buộc thiết kế kịch bản:** Một số tổ hợp đặc trưng không độc lập (ví dụ `time = 7AM` gắn liền với `destination = Work`; `No Urgent Place` không có `direction_same = 1` hay `>=25 min`), đòi hỏi phân tầng kiểm soát confounder thay vì gộp thô.
  4. **Không suy diễn nhân quả:** Dữ liệu khảo sát quan sát chỉ cho phép kết luận **mối liên hệ**, không chứng minh tác động nhân quả.

### 1.4 Đối chiếu các phân tích công khai (`docs/related_work.md`)
- Nhóm đã rà soát bài báo gốc (Wang et al., 2017) cùng các repo tham khảo công khai như `seetharamanr86/customer_coupon_acceptance` và `arnavkundu/MLEngineerNanoDegree-Capstone`. Điểm khác biệt của nhóm là kết hợp kiểm soát confounder có hệ thống (D4), khoảng tin cậy bootstrap 95% (D5), kiểm tra độ bền (D6) và pipeline ML chống data leakage (chi tiết xem [`docs/related_work.md`](../docs/related_work.md)).

## 2. Load & Overview
**TODO:** Số dòng/cột, ý nghĩa mỗi dòng, tên cột, `dtypes`, cột cần biến đổi, nhãn Y.

In [ ]:
# Thiết lập đường dẫn (chạy Jupyter từ repo root hoặc thư mục notebooks đều được)
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from coupon_project.data_io import load_raw_data, row_count_log
from coupon_project.quality import missing_table

df = load_raw_data(ROOT / "data" / "raw" / "in-vehicle-coupon-recommendation.csv")
print(f"Raw data: {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
# Khởi đầu việc hiểu dữ liệu (không thay đổi dữ liệu gốc)
display(df.head())
display(df.dtypes.to_frame("dtype"))
print("Duplicates:", df.duplicated().sum())
print("Unique target values:", df["Y"].unique())

## 3. Missing data
**TODO:** Hiển thị số lượng và %, kiểm tra missing có theo nhóm không; nêu quyết định xử lý từng cột. Đặc biệt xem `car`, `Bar`, `CoffeeHouse`, `CarryAway`, `RestaurantLessThan20`, `Restaurant20To50`.

In [ ]:
display(missing_table(df))
# TODO: biểu đồ missingness theo các cột quan trọng và kiểm tra pattern.

## 4. Numerical & categorical exploration
**TODO:** Trình bày phân phối, min/max hợp lý, nhóm ít xuất hiện, nhãn không nhất quán. Không ép `age`, `income` thành số thô. Kiểm tra `toCoupon_GEQ5min` có phải cột hằng không. Chọn biểu đồ phù hợp, không vẽ máy móc một biểu đồ/cột.

In [ ]:
# TODO: kiểm tra phân phối và value_counts phù hợp từng nhóm cột.
# Gợi ý: df["coupon"].value_counts(dropna=False)
# Gợi ý: df["age"].value_counts(dropna=False)

## 5. Relationships
**TODO:** Cross-tab theo coupon, expiration, destination, passanger, Y; quan hệ các cờ khoảng cách/hướng; lưu ý các tổ hợp ít mẫu.

In [ ]:
# TODO: phân tích quan hệ biến và biểu đồ; không kết luận nếu chưa kiểm tra số mẫu.

## 6. Preprocessing decisions (ghi trước khi làm)

| Thao tác | Lý do | Rows trước | Rows sau | Ảnh hưởng/rủi ro |
| --- | --- | --- | --- | --- |
| TODO | TODO | TODO | TODO | TODO |

**TODO:** Nếu tạo dữ liệu processed, hãy ghi từng bước đổi số dòng bằng `row_count_log(before, after, action)`. Không tự động dropna mọi cột.

## 7. Key observations (3–5), cleaning plan & follow-up questions

**TODO:** 3–5 nhận xét có số liệu, quality issues nghiêm trọng, kế hoạch làm sạch, patterns dẫn tới Q1–Q6. Tránh dùng suy đoán hoặc kết quả của notebook khác.